In [35]:
import torch
from torch.utils.data import DataLoader, random_split, TensorDataset
import logging
from torch import nn 

%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from datasets import TinyShakeSpeareDataset
from tokenizer import SimpleTokenizer

dataset = TinyShakeSpeareDataset('data/tinyshakespeare.txt')
text = dataset.load()

tokenizer = SimpleTokenizer()
tokenizer.construct(text)

In [ ]:




data = torch.tensor([stoi[w] for w in words], dtype=torch.long)

# We need to create "chunks of text" 
seq_len = 64
n = data.shape[0]
X_list, Y_list = list(), list()
for i in range(0, len(data) - seq_len, seq_len):
    x = data[i: i + seq_len]
    y = data[i + 1: i + seq_len + 1]

    X_list.append(x)
    Y_list.append(y)


X = torch.stack(X_list, dim=0)
Y = torch.stack(Y_list, dim=0)


batch_size = 8
dataset = TensorDataset(X, Y)
train_size = int(.80*len(dataset))
test_size = int(.10*len(dataset))
val_size = len(dataset) - train_size - test_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, test_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  drop_last=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,  drop_last=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,  drop_last=True, pin_memory=True)



print(f"X tensor shape {X.shape}")
print(f"Y tensor shape {Y.shape}")
print(f"vocab size:  {vocab_size}")
assert torch.equal(X[:, 1:seq_len], Y[:, :seq_len - 1]), "Label != Next Token"

print(f"len(train_loader): {len(train_loader)}")
print(f"len(val_loader): {len(val_loader)}")
print(f"len(test_loader): {len(test_loader)}")



X tensor shape torch.Size([2654, 64])
Y tensor shape torch.Size([2654, 64])
vocab size:  42197
len(train_loader): 265
len(val_loader): 33
len(test_loader): 33


In [24]:
class RMSNorm(nn.Module):

    def __init__(self, d_model):
        super().__init__()
        self.epsilon = 1e-6
        self.gamma = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor):
        mean = torch.mean(torch.pow(x, 2),dim=-1, keepdim=True)
        rms = torch.sqrt(mean + self.epsilon)
        rms_norm = (x / rms) * self.gamma
        return rms_norm


norm = RMSNorm(2)
norm.epsilon = 0


x = torch.tensor(
    [
        [[1, 1], [2, 2], [3, 3], [4, 4]],
        [[5, 5], [6, 6], [7, 7], [8, 8]]
    ], dtype=torch.float
)

expected = torch.tensor(
    [
        [[1, 1], [1, 1], [1, 1], [1, 1]],
        [[1, 1], [1, 1], [1, 1], [1, 1]],
    ], dtype=torch.float
)

o = norm(x)
assert torch.equal(o, expected), f"{o} != {expected}"


In [25]:

# x = torch.tensor([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]], dtype=torch.float)

# seq_len, d_model = x.shape[0], x.shape[1]


# # Notes: 
# # m - This value controls how much we rotate the embedding and it is depedent on the token's
# #     location in the sequence. Token's later in the sequency are rotated furhter. 
# # theta - This value controls the frequency at which we rotate. This value is predetermined, 
# #         and is depedent only on d_model and the RoPE base frequency (10k). Theta is calculated 
# #         for d_model/2 pairs and pairs that are near the beginning of the embedding rotate 
# #         have 


# i = torch.arange(0, d_model, 2, dtype=torch.float)
# theta = torch.pow(10000, -i/d_model)
# m = torch.arange(0, seq_len, dtype=torch.float)
# rope_angles = m.unsqueeze(-1) * theta
# print(f"rope_angles={rope_angles.shape}")


# cos_raw = torch.cos(rope_angles)
# sin_raw = torch.sin(rope_angles)
# print(f"cos_raw={cos_raw.shape}")
# print(f"sin_raw={sin_raw.shape}")

# cos_expaned = torch.repeat_interleave(cos_raw, dim=1)
# sin_expaned = torch.repeat_interleave(sin_raw, dim=1)
# cos_expaned = torch.repeat_interleave(cos_raw, repeats=2, dim=1)
# sin_expaned = torch.repeat_interleave(sin_raw, repeats=2, dim=1)

# print(f"cos_expaned={cos_expaned.shape}")
# print(f"sin_expaned={sin_expaned.shape}")


# x_rotated = torch.rotate_half(x)

# # R = torch.stack(
# #     [
# #         torch.stack([
# #             torch.cos(rope_angles), -torch.sin(rope_angles)
# #         ], dim=-1),
# #         torch.stack([
# #             torch.sin(rope_angles), torch.cos(rope_angles)
# #         ], dim=-1)
# #     ], dim=-1
# # )


# # print(rope_angles.shape)

# # # print(R)
# # print(R.shape)


# # # cos = torch.cos(rope_angles)
# # # sin = torch.sin(rope_angles)
# # x_grouped = x.view(4, 2, 2).unsqueeze(-1)



# # # print(R.shape)
# # # print(x_grouped.shape)
# # x_rotated = (R @ x_grouped)

# # print(x_rotated.shape)

# # print(x_rotated)

# # print(x_rotated.shape)

# # We have input of x of size 4 tokens with dim = 4

# # We need to rotate each token's embedding 










In [26]:
import math 
class AttentionHead(nn.Module):

    def __init__(self, d_model, max_seq_len=512):
        super().__init__()
        self.d_model = d_model
        # in and out set o d_model since this is a single head. 
        self.Q = torch.nn.Linear(d_model, d_model)
        self.K = torch.nn.Linear(d_model, d_model)
        self.V = torch.nn.Linear(d_model, d_model)

        mask = torch.triu(
            torch.ones((max_seq_len, max_seq_len), dtype=torch.bool),
            diagonal=1
        )
        self.register_buffer('mask', mask)

    def forward(self, x):
        q = self.Q(x)
        k = self.K(x)
        v = self.V(x)

        seq_len = x.shape[-2] #... x seq_len x dim_model
        qk = q @ k.transpose(-2, -1)
        qk_masked = qk.masked_fill(self.mask[:seq_len, :seq_len], -torch.inf)
        o = torch.softmax(qk_masked / math.sqrt(self.d_model), dim=-1) @ v
        return o

In [27]:

from transformer import Transformer

# Create embedding. 

class FFN(nn.Module):

    def __init__(self, seq_len, d_model):
        super(FFN, self).__init__()
        self.seq = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model),
        )

    def forward(self, x):
        return self.seq(x)



class BabyTransformer(Transformer):
    '''
    Embeddings, no position information, 
    single-head scaled dot-product attention
    '''

    def __init__(self, itos: dict, stoi: dict, seq_len: int, vocab_size: int, d_model: int):
        super().__init__(itos, stoi, seq_len, vocab_size, d_model)


        # [B x seq_len x d_model]
        self.embedding = nn.Embedding(vocab_size, d_model)
        # [B x seq_len x d_model]
        self.prenorm_1 = RMSNorm(d_model)
        self.head = AttentionHead(d_model, seq_len)
        self.prenorm_2 = RMSNorm(d_model)
        self.ffn = FFN(seq_len, d_model)
        self.prenorm_3 = RMSNorm(d_model)
        self.linear = torch.nn.Linear(d_model, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        x_norm = self.prenorm_1(emb)
        head_o = self.head(x_norm) + emb
        head_o_norm = self.prenorm_2(head_o)
        ffn_o = self.ffn(head_o_norm) + head_o
        ffn_o_norm = self.prenorm_3(ffn_o)
        logits = self.linear(ffn_o_norm) #+ ffn_o
        return logits 


In [28]:
from torch import optim

d_model = 16
epochs = 1
max_seq_len = 64

model = BabyTransformer(itos, stoi, max_seq_len, vocab_size, d_model)


def evaluate(model, val_loader, criterion, device, max_batches=None):
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_idx, (batch_x, batch_y) in enumerate(val_loader):
            if max_batches is not None and batch_idx >= max_batches:
                break 

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            logits = model(batch_x)
            B, S, d_out = logits.shape

            all_seqs = logits.view(B*S, d_out)
            y = batch_y.view(B*S)
            loss = criterion(all_seqs, y)
            val_loss += loss.item()

    return val_loss / len(val_loader)



def train_loop(model, train_loader, val_loader=None, num_epochs=5, lr=0.001):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.1,
        patience=3,
    )
    
    print(
        f"Initialized {model.__class__.__name__} | "
        f"vocab_size={model.vocab_size}, d_model={model.d_model}, seq_len={model.seq_len}"
    )
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for batch_idx, (batch_x, batch_y) in enumerate(train_loader):

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            logits = model(batch_x)
            B, S, d_out = logits.shape

            all_seqs = logits.view(B*S, d_out)
            y = batch_y.view(B*S)
            loss = criterion(all_seqs, y)

            if batch_idx % 100 == 0:
                logging.info(
                    f"Epoch {epoch+1}/{num_epochs} | "
                    f"Batch {batch_idx}/{len(train_loader)} | "
                    f"Loss: {loss.item():.4f}"
                )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        epoch_train_loss = running_loss / len(train_loader)
        history["train_loss"].append(epoch_train_loss)


        # Validation evaluation
        if val_loader is not None:
            epoch_val_loss = evaluate(model, val_loader, criterion, device)
            history["val_loss"].append(epoch_val_loss)
            scheduler.step(epoch_val_loss)
            logging.info(
                f"--> Epoch {epoch+1}/{num_epochs} Complete | "
                f"Train Loss: {epoch_train_loss:.4f} | "
                f"Val Loss: {epoch_val_loss:.4f}"
            )
        else:
            scheduler.step(epoch_train_loss)
            logging.info(
                f"--> Epoch {epoch+1}/{num_epochs} Complete | "
                f"Train Loss: {epoch_train_loss:.4f}"
            )

train_loop(model, train_loader, val_loader, num_epochs=10)

Initialized BabyTransformer | vocab_size=42197, d_model=16, seq_len=64


2026-09-19 10:11:06,071 - INFO - Epoch 1/10 | Batch 0/265 | Loss: 10.8401
2026-09-19 10:11:06,440 - INFO - Epoch 1/10 | Batch 100/265 | Loss: 9.3493
2026-09-19 10:11:06,772 - INFO - Epoch 1/10 | Batch 200/265 | Loss: 8.1744
2026-09-19 10:11:07,033 - INFO - --> Epoch 1/10 Complete | Train Loss: 9.2628 | Val Loss: 8.2741
2026-09-19 10:11:07,035 - INFO - Epoch 2/10 | Batch 0/265 | Loss: 7.8706
2026-09-19 10:11:07,354 - INFO - Epoch 2/10 | Batch 100/265 | Loss: 7.8416
2026-09-19 10:11:07,680 - INFO - Epoch 2/10 | Batch 200/265 | Loss: 7.8298
2026-09-19 10:11:07,927 - INFO - --> Epoch 2/10 Complete | Train Loss: 7.9220 | Val Loss: 8.3366
2026-09-19 10:11:07,929 - INFO - Epoch 3/10 | Batch 0/265 | Loss: 7.6194
2026-09-19 10:11:08,251 - INFO - Epoch 3/10 | Batch 100/265 | Loss: 7.8549
2026-09-19 10:11:08,571 - INFO - Epoch 3/10 | Batch 200/265 | Loss: 7.7800
2026-09-19 10:11:08,822 - INFO - --> Epoch 3/10 Complete | Train Loss: 7.8233 | Val Loss: 8.4055
2026-09-19 10:11:08,824 - INFO - Epoch 

In [29]:
import os
import re
import time
import math
import torch
import torch.nn as nn
from torch import optim
import pandas as pd


class BenchmarkHarness:
    def __init__(self, train_loader, val_loader, total_steps=500, lr=1e-3, seed=42):
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.total_steps = total_steps
        self.lr = lr
        self.seed = seed
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.criterion = nn.CrossEntropyLoss(ignore_index=0)
        self.results = []

    def _reset_seed(self):
        torch.manual_seed(self.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.seed)

    def _evaluate(self, model, max_batches=20):
        model.eval()
        total_loss = 0.0
        total_batches = 0
        with torch.no_grad():
            for idx, (batch_x, batch_y) in enumerate(self.val_loader):
                if max_batches and idx >= max_batches:
                    break
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)
                logits = model(batch_x)
                B, S, V = logits.shape
                loss = self.criterion(logits.view(B * S, V), batch_y.view(B * S))
                total_loss += loss.item()
                total_batches += 1
        return total_loss / max(1, total_batches)

    def run_candidate(self, name: str, model_builder_fn, metadata: dict = None):
        self._reset_seed()
        model = model_builder_fn().to(self.device)
        total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

        optimizer = optim.AdamW(model.parameters(), lr=self.lr, weight_decay=0.01)
        data_iter = iter(self.train_loader)

        best_val_loss = float("inf")
        start_time = time.time()

        print(f"--> Benchmarking: {name} ({total_params:,} params)")

        for step in range(1, self.total_steps + 1):
            try:
                batch_x, batch_y = next(data_iter)
            except StopIteration:
                data_iter = iter(self.train_loader)
                batch_x, batch_y = next(data_iter)

            model.train()
            batch_x = batch_x.to(self.device)
            batch_y = batch_y.to(self.device)

            logits = model(batch_x)
            B, S, V = logits.shape
            loss = self.criterion(logits.view(B * S, V), batch_y.view(B * S))

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            if step % 50 == 0 or step == self.total_steps:
                val_loss = self._evaluate(model)
                best_val_loss = min(best_val_loss, val_loss)

        wall_time = time.time() - start_time
        final_val_loss = self._evaluate(model)
        val_ppl = math.exp(final_val_loss) if final_val_loss < 20 else float("inf")

        run_entry = {
            "Model": name,
            "Positional Emb": metadata.get("pos_emb", "None") if metadata else "None",
            "Attention Type": metadata.get("attn_type", "Single-Head") if metadata else "Single-Head",
            "Norm": metadata.get("norm", "None") if metadata else "None",
            "Parameters": f"{total_params:,}",
            "Best Val Loss": round(best_val_loss, 4),
            "Final Val Loss": round(final_val_loss, 4),
            "Final PPL": round(val_ppl, 1),
            "Steps/sec": round(self.total_steps / wall_time, 2),
        }
        self.results.append(run_entry)

    def summary(self, readme_path="../README.md"):
        df = pd.DataFrame(self.results).sort_values(by="Best Val Loss")
        md_table = df.to_markdown(index=False)

        # 1. Print directly to terminal/cell output
        print("\n" + "=" * 35 + " BENCHMARK LEADERBOARD " + "=" * 35)
        print(md_table)
        print("=" * 93 + "\n")

        # 2. Write or update README.md cleanly using markers
        start_marker = "<!-- BENCHMARK_START -->"
        end_marker = "<!-- BENCHMARK_END -->"
        section_content = f"{start_marker}\n## Model Benchmark Results\n\n{md_table}\n{end_marker}"

        if os.path.exists(readme_path):
            with open(readme_path, "r", encoding="utf-8") as f:
                content = f.read()

            if start_marker in content and end_marker in content:
                # Replace existing benchmark table in place
                pattern = rf"{re.escape(start_marker)}.*?{re.escape(end_marker)}"
                updated_content = re.sub(pattern, section_content, content, flags=re.DOTALL)
            else:
                # Append to existing README
                updated_content = content.rstrip() + f"\n\n{section_content}\n"
        else:
            # Create fresh README
            updated_content = f"# Transformer Experiments\n\n{section_content}\n"

        with open(readme_path, "w", encoding="utf-8") as f:
            f.write(updated_content)

        print(f"Benchmark results successfully written to {readme_path}")
        return df

In [30]:
harness = BenchmarkHarness(train_loader, val_loader, total_steps=300)

harness.run_candidate(
    name="Baseline",
    model_builder_fn=lambda: BabyTransformer(stoi, itos, max_seq_len, vocab_size, d_model),
    metadata={"pos_emb": "None", "attn_type": "Single-Head Causal", "norm": "RMSNorm"}
)

# Prints Markdown to stdout and updates README.md
harness.summary()

--> Benchmarking: Baseline (1,395,493 params)

=================================== BENCHMARK LEADERBOARD ===================================
| Model    | Positional Emb   | Attention Type     | Norm    |   Parameters |   Best Val Loss |   Final Val Loss |   Final PPL |   Steps/sec |
|:---------|:-----------------|:-------------------|:--------|-------------:|----------------:|-----------------:|------------:|------------:|
| Baseline | None             | Single-Head Causal | RMSNorm |    1,395,493 |          8.2535 |           8.2535 |        3841 |      253.65 |

Benchmark results successfully written to ../README.md


,Model,Positional Emb,Attention Type,Norm,Parameters,Best Val Loss,Final Val Loss,Final PPL,Steps/sec
0,Baseline,None,Single-Head Causal,RMSNorm,"1,395,493",8.2535,8.2535,3841.0,253.65


In [31]:

model.generate(context="the king shall set you free")


NameError: name 'stoi' is not defined